### Introduction
O2

| Chr | Start   | End    |
|-----|---------|--------|
| chr7   | 10953483  | 10953507 |

In [1]:
import pysam
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

In [2]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True).eval()
    return tokenizer, model

In [3]:
genome_file = '/workdir/jz963/genomes/B73_v5/Zm-B73-REFERENCE-NAM-5.0.fa'
fasta = pysam.FastaFile(genome_file)

In [4]:
start = 10953483
end = 10953507
chrom = 'chr7'
del_len = end - start + 1
del_len

25

In [38]:
del_seq = fasta.fetch(chrom, start - 1, end)

In [46]:
del_seq

'CATGGACGGAGAAGTAGAGATTCTG'

In [40]:
len(del_seq)

25

In [5]:
del_len // 2

12

In [6]:
ref_seq = fasta.fetch(chrom, start - 1 - 4096, end + 4096)
mut_seq = ref_seq[0:4096] + ref_seq[-4096:]
ref_seq = ref_seq[(del_len//2):-(del_len//2)][0:8192]

In [7]:
tokenizer, model = load_model('kuleshov-group/compo-cad2-l48-d1536-dna-chtk-c8k-1t-v1-b2-lr4e4-NzqiLr')
model.to('cuda:0')

/home/jz963/miniconda3/envs/transformers/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


CaduceusForMaskedLM(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): RCPSEmbedding(
          (embedding): Embedding(8, 1536)
        )
      )
      (layers): ModuleList(
        (0-47): 48 x RCPSMambaBlock(
          (mixer): RCPSWrapper(
            (submodule): BiMambaWrapper(
              (mamba_fwd): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
                (norm): RMSNorm()
                (out_proj): Linear(in_features=3072, out_features=1536, bias=False)
              )
              (mamba_rev): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
   

In [8]:
model.to(torch.bfloat16)

CaduceusForMaskedLM(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): RCPSEmbedding(
          (embedding): Embedding(8, 1536)
        )
      )
      (layers): ModuleList(
        (0-47): 48 x RCPSMambaBlock(
          (mixer): RCPSWrapper(
            (submodule): BiMambaWrapper(
              (mamba_fwd): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
                (norm): RMSNorm()
                (out_proj): Linear(in_features=3072, out_features=1536, bias=False)
              )
              (mamba_rev): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
   

In [9]:
inputs = tokenizer(
            [ref_seq, mut_seq],
            truncation=False,
            padding=False,
            return_tensors="pt",
            return_attention_mask=False,
            return_token_type_ids=False,)['input_ids'].to(model.device)

In [10]:
inputs.shape

torch.Size([2, 8192])

In [11]:
with torch.no_grad():
    outputs = model(inputs)

In [12]:
nucleotides = list('acgt')
logits = outputs.logits[..., [tokenizer.get_vocab()[nc] for nc in nucleotides]]
probs = torch.nn.functional.softmax(logits, dim=2).cpu().numpy()
probs.shape

(2, 8192, 4)

In [13]:
probs[0][0]

array([0.9037083 , 0.01192195, 0.02815589, 0.05621383], dtype=float32)

In [22]:
flank_len = (8192-del_len)//2

In [23]:
ref_prob_left = probs[0][0:(flank_len+1)][-10:]
mut_prob_left = probs[1][0:4096][-10:]

ref_prob_right = probs[0][-flank_len:][0:10]
mut_prob_right = probs[1][-4096:][0:10]

In [24]:
ref_prob = np.concatenate((ref_prob_left, ref_prob_right), axis = 0)
mut_prob = np.concatenate((mut_prob_left, mut_prob_right), axis = 0)
seq = ref_seq[0:flank_len][-10:] + ref_seq[-flank_len:][0:10]

In [25]:
ref_prob.shape, mut_prob.shape, len(seq)

((20, 4), (20, 4), 20)

In [30]:
scores = []
nucleotides = "ACGT"
for idx, nt in enumerate(seq):
    if nt in nucleotides:
        refProb = ref_prob[idx, nucleotides.index(nt)]
        mutProb = mut_prob[idx, nucleotides.index(nt)]
        print(idx, refProb, mutProb)
        scores.append(np.log(mutProb / refProb))
    else:
        scores.append(0)

0 0.00014709326 0.0030584952
1 0.0123096155 0.044338614
2 0.00033393316 0.007855003
3 0.00041047874 0.017336473
4 0.010791882 0.111642025
5 0.0005591473 0.023052417
6 0.0008760019 0.03497766
7 0.49349937 0.48988393
8 0.00029015966 0.03027776
9 0.00098469 0.030467069
10 0.9925708 0.64269876
11 0.9546333 0.7294935
12 0.83538085 0.6602638
13 0.9796591 0.5730842
14 0.9333764 0.77184874
15 0.9569346 0.91312957
16 0.996881 0.8213339
17 0.96633774 0.8753348
18 0.89531124 0.7188726
19 0.98273057 0.8906198


In [28]:
scores

[3.0346115,
 1.2814752,
 3.1579652,
 3.7432437,
 2.3365033,
 3.719113,
 3.6870966,
 -0.007353094,
 4.6477375,
 3.4320748,
 -0.43462226,
 -0.2689768,
 -0.2352483,
 -0.53617203,
 -0.1900199,
 -0.04685722,
 -0.19370174,
 -0.09890696,
 -0.21948725,
 -0.09841738]

In [27]:
np.mean(scores)

1.3355029

## The quantile of simulated deletions  using flanking 20bp (+10/-10)

| Percentile | Value      |
|------------|------------|
| 0.1%       | -1.6905446 |
| 1%         | -1.1289620 |
| 10%        | -0.4855522 |
| 50%        | -0.0777035 |

## Conclusion
Unfortunately, this insertion doesn't fall into the top 10%

In [35]:
import h5py
with h5py.File('O2_logits.h5', 'a') as hf:
    model_logits = torch.tensor(hf['predicted_logits'][:])
    true_ids = torch.tensor(hf['true_token_ids'][:])

In [34]:
model_logits.shape

torch.Size([27, 8])

In [36]:
nucleotides = list('acgt')
model_logits = model_logits[:, [tokenizer.get_vocab()[nc] for nc in nucleotides]] # only retain atcg
model_logits = model_logits.clone().detach()
model_probs = torch.nn.functional.softmax(model_logits, dim=1).numpy()

In [41]:
model_probs.shape

(27, 4)

In [45]:
for idx, base in enumerate(del_seq):
    prob = model_probs[idx, nucleotides.index(base.lower())]
    print(prob)

0.124617085
0.15864497
0.24709289
0.23195376
0.71650326
0.69659615
0.217264
0.65154994
0.27538145
0.23444416
0.6702533
0.5557795
0.22219318
0.58240557
0.09290053
0.21951433
0.4310464
0.32146057
0.37119344
0.50546676
0.11811329
0.2741491
0.11510108
0.16726288
0.37705266


In [43]:
nucleotides

['a', 'c', 'g', 't']

In [48]:
logits = np.loadtxt('O2-single-BP-logit.tsv', delimiter='\t')
logits

array([[7.36016408e-02, 7.81560361e-01, 7.81642869e-02, 6.66736811e-02],
       [9.64526236e-01, 1.08486386e-02, 1.45252459e-02, 1.00998506e-02],
       [9.08627757e-04, 1.50349773e-02, 4.28772531e-04, 9.83627677e-01],
       [4.08759899e-02, 3.60015601e-01, 5.39746284e-01, 5.93620576e-02],
       [1.01687741e-02, 7.79034337e-04, 9.88805532e-01, 2.46557087e-04],
       [9.94477332e-01, 1.16864848e-03, 3.97288753e-03, 3.81156278e-04],
       [2.13118270e-01, 3.24526757e-01, 1.01757407e-01, 3.60597581e-01],
       [3.27768247e-03, 1.82683649e-03, 9.94699836e-01, 1.95653018e-04],
       [1.59797445e-02, 7.36170681e-03, 9.74128127e-01, 2.53039622e-03],
       [4.69781220e-01, 2.22564891e-01, 1.14634432e-01, 1.93019435e-01],
       [8.61858600e-04, 1.01836352e-02, 9.88857329e-01, 9.72557609e-05],
       [9.02166069e-01, 6.04318455e-02, 2.20344588e-02, 1.53676327e-02],
       [2.26522073e-01, 1.66594461e-02, 7.45422602e-01, 1.13958316e-02],
       [1.62793204e-01, 1.31441250e-01, 6.93573236e

In [49]:
for idx, base in enumerate(del_seq):
    prob = logits[idx, nucleotides.index(base.lower())]
    print(prob)

0.7815603613853455
0.9645262360572815
0.9836276769638062
0.5397462844848633
0.9888055324554443
0.9944773316383362
0.32452675700187683
0.9946998357772827
0.9741281270980835
0.46978121995925903
0.9888573288917542
0.9021660685539246
0.2265220731496811
0.6935732364654541
0.03223923593759537
0.33196666836738586
0.8234775066375732
0.330326646566391
0.7023189663887024
0.6047079563140869
0.6120108962059021
0.2964455783367157
0.5318960547447205
0.13647302985191345
0.19611361622810364
